# Algorithmic Hiring - Curso Introductorio

## Práctica Parte 2: FA*IR

La segunda sesión práctica del curso introductorio al Algorithmic Hiring tiene como objetivo implementar en Python el algoritmo [`FA*IR`](https://arxiv.org/pdf/1706.06368) para crear rankings más equitativos. La idea es intervenir la lista ordenada de candidatos generada en la sesión práctica I para hacerla más representativa en los primeros 10 lugares, que son normalmente los que reciben más atención de los reclutadores.

> **Completar estos datos antes de entregar**

Nombre: <font color="blue">Pol Mazón Caballero</font>

Email: <font color="blue">pol.mazon@gmail.com</font>

Fecha: <font color="blue">31-07-2026</font>

## 1. Importar datos

In [ ]:
# [NO MODIFICAR] carga librerias requeridas
import numpy as np
import os
import pandas as pd

from google.colab import drive

In [ ]:
# [NO MODIFICAR] otorga permisos al notebook para importar datos del drive
drive.mount('/content/drive')

In [ ]:
# indicar ruta a la carpeta donde se encuentra el dataset
data_dir = '/content/drive/MyDrive/'

In [ ]:
# carga los datos generados al final de la practica 1: la lista de
# candidatos de la vacante de ventas ya ordenada por el modelo LTR
sales_ordered_df = pd.read_csv(os.path.join(data_dir, 'sales_ordered_candidates.csv'))

In [ ]:
# muestra la cantidad de registros en el dataset
print(f'El dataset contiene {sales_ordered_df.shape[0]} candidatos')

In [ ]:
# visualiza los 10 primeros registros del dataset
sales_ordered_df.head(10)

**Explicación de las variables**

* `exp_months`: indica la experiencia profesional del candidato acumulada en meses
* `has_pro_exp`: indica si el candidato tiene la cantidad de experiencia profesional requerida por el puesto; 1 la tiene, 0 no la tiene
* `occupied_role`: indica si el candidato ocupó el rol vacante en algún momento durante su carrera profesional; 1 ocupó el rol; 0 no lo ocupó
* `num_job_skills`: indica el número de habilidades requeridas por el puesto que el candidato declaró tener
* `gender_c`: indica el género del candidato; M masculino, F Femenino
* `origin_c`: indica la procedencia del candidato; EU Europeo, Non-EU no Europeo
* `score`: indica qué tan bien el perfil del candidato encaja con los requerimientos del puesto; cuanto más cerca de 1, mejor encaje

## 2. Algoritmo FA*IR

Primero definimos la tabla que contiene el número mínimo de candidatos del
grupo protegido para cada posición del ranking dada una `proporción` de `0.7`.
Los valores de la tabla fueron extraídos de la `Tabla 2` del artículo [FA*IR: A Fair Top-k Ranking Algorithm](https://arxiv.org/pdf/1706.06368).

In [ ]:
# [NO MODIFICAR] crea la tabla con el número mínimo de candidatos del
# grupo protegido para cada posición del ranking dada un proporción de 0.7
M_TABLE = [0, 1, 1, 2, 2, 3, 3, 4, 5, 5]

El segundo paso consiste en separar los candidatos del grupo protegido (no europeos) de los candidatos del grupo no protegido (europeos).

<font color="red">Utiliza la celda de abajo para colocar en un dataset los candidatos no europeos y en otra los candidatos europeos.</font>

In [ ]:
# separa los candidatos no europeos (grupo protegido) de los candidatos
# europeos (grupo no protegido), manteniendo el orden por encaje (score)
protected_df = (
    sales_ordered_df[sales_ordered_df['origin_c'] == 'Non-EU']
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)
unprotected_df = (
    sales_ordered_df[sales_ordered_df['origin_c'] == 'EU']
    .sort_values('score', ascending=False)
    .reset_index(drop=True)
)

print(f'Candidatos del grupo protegido (Non-EU): {protected_df.shape[0]}')
print(f'Candidatos del grupo no protegido (EU): {unprotected_df.shape[0]}')

El tercer paso consiste en implementar el algoritmo `FA*IR`. Básicamente, consiste en agregar candidatos del grupo protegido si el número de candidatos en la posición actual del ranking equitativo es menor que el establecido en la tabla `M_TABLE` o bien agregar el candidato con el mejor `encaje` considerando ambos grupos (protegidos y no protegidos).

<font color="red">Utiliza la celda de abajo para implementar el algoritmo `FA*IR`.</font>

In [ ]:
# implementa el algoritmo FA*IR
def fair_top_k(protected_df, unprotected_df, m_table, k=10, score_col='score'):
  """Construye un top-k equitativo combinando candidatos protegidos y no
  protegidos, garantizando en cada posición el minimo de candidatos del
  grupo protegido indicado en m_table."""
  fair_ranking = []
  p_idx = 0
  u_idx = 0
  n_protected = 0

  for i in range(k):
    protected_available = p_idx < len(protected_df)
    unprotected_available = u_idx < len(unprotected_df)

    # si aun no se alcanzo el minimo de candidatos protegidos exigido para
    # esta posicion, se agrega el siguiente candidato protegido
    if n_protected < m_table[i] and protected_available:
      fair_ranking.append(protected_df.iloc[p_idx])
      p_idx += 1
      n_protected += 1
    # en caso contrario, se agrega el candidato con mejor encaje entre
    # ambos grupos
    elif protected_available and unprotected_available:
      if protected_df.iloc[p_idx][score_col] >= unprotected_df.iloc[u_idx][score_col]:
        fair_ranking.append(protected_df.iloc[p_idx])
        p_idx += 1
        n_protected += 1
      else:
        fair_ranking.append(unprotected_df.iloc[u_idx])
        u_idx += 1
    elif protected_available:
      fair_ranking.append(protected_df.iloc[p_idx])
      p_idx += 1
      n_protected += 1
    elif unprotected_available:
      fair_ranking.append(unprotected_df.iloc[u_idx])
      u_idx += 1
    else:
      break

  return pd.DataFrame(fair_ranking).reset_index(drop=True)

fair_top10_df = fair_top_k(protected_df, unprotected_df, M_TABLE, k=10, score_col='score')

Por último, <font color="red">utiliza la celda de abajo para mostrar cómo queda el top-10 de candidatos luego de aplicar el algoritmo `FA*IR`.</font>

In [ ]:
# visualiza el top-10 de candidatos luego del algoritmo FA*IR
fair_top10_df

<font color="red">Comentario sobre el efecto de aplicar el algoritmo `FA*IR`</font>

Al comparar el top-10 original (`sales_ordered_df.head(10)`, ordenado únicamente por el `score` predicho por el modelo de la práctica 1) con el top-10 producido por `FA*IR`, se observa que este último incorpora candidatos del grupo protegido (`Non-EU`) que en el ranking original no llegaban a figurar entre los 10 primeros puestos, a pesar de tener un encaje (`score`) comparable al de varios candidatos europeos que sí aparecían en el top-10. Esto ocurre porque el ranking original, al estar determinado solo por el `score`, puede reproducir patrones históricos de contratación que favorecen sistemáticamente a un grupo, aunque existan candidatos igualmente cualificados en el grupo menos representado.

El algoritmo `FA*IR` corrige esta situación garantizando, posición a posición, una representación mínima del grupo protegido (según la tabla `M_TABLE`, calculada para una proporción del 70%), sin descartar a los candidatos con mejor encaje: en cada paso sigue eligiendo, entre los candidatos disponibles de ambos grupos, al de mejor `score`, salvo cuando es necesario incorporar un candidato protegido para cumplir el mínimo exigido. El resultado es una lista final que sigue priorizando la calidad del encaje, pero que evita que el top-10 quede compuesto casi en su totalidad por un solo grupo demográfico.

En mi opinión, incorporar candidatos de grupos menos favorecidos de esta manera es una intervención razonable y necesaria: no se trata de rebajar el estándar de calidad (los candidatos añadidos siguen cumpliendo con los requisitos del puesto y tienen un encaje competitivo), sino de corregir un sesgo estructural que, de no intervenirse, dejaría sistemáticamente fuera del proceso de selección a personas igualmente cualificadas solo por pertenecer a un grupo minoritario. Aun así, es importante ser transparente con los reclutadores sobre el uso de este tipo de reordenamientos y monitorear que la intervención no perjudique injustamente a candidatos individuales del grupo no protegido con un encaje claramente superior.

## Entrega (individual)

Para entregar la práctica, seguir las siguientes instrucciones:

*   Descargar el notebook `File->Download->Download .ipynb`
*   Renombrar a `nombre.apellido-practica2.ipynb`
*   Enviar a través del `ECampus` a más tardar el <font color="red"><b>28-05-2026 23:59</b></font>.

<font size="+2" color="blue">Declaro que, excepto el código provisto por el instructor del curso, todo el resto del código, texto y figuras fueron producidos por mí mismo.</font>